# Task 3. Preprocessing Pipeline

### Explainable AI Credit Risk Decision Platform : Integrating Structured Borrower Data, NLP-Driven Text Intelligence, and Macroeconomic Indicators for Transparent Lending Decisions.

#### Reason:
#### This process is to clean, formats and transform the raw data into structured numerical representation so machine learning models can learn patterns accurately and efficiently.


In [2]:
# PREPROCESSING PIPELINE
# ___________________________________________________________

# Data manipulation

import os 
import pandas as pd
import numpy as np

# Save models

import joblib
from scipy.sparse import save_npz

# nlp

import re
import string

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Train/Test split

from sklearn.model_selection import train_test_split

# Preprocessing

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer

 # Create folders

from pathlib import Path

In [3]:
# Creating directories
# ____________________________________________________________

try:
    Path("models").mkdir(parents=True, exist_ok=True)
    
    Path("data/processed").mkdir(parents=True, exist_ok=True)
    
    print("Directories created in current directory!")
    
except Exception as e:
    
    print(f"Error: {e}")

Directories created in current directory!


In [4]:
# Data Overview
# _________________________________________

# finding the folder the notebook is currently running in

current_folder = Path('.').resolve()

# Define the path pointing to your home directory folder

file_path = current_folder / "data" / "processed" / "lendingclub_clean.csv"

# Read the data

df = pd.read_csv(file_path, low_memory=False)

print(f"Dataset Shape: {df.shape}")

df.head()

Dataset Shape: (500000, 151)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Binary Target
# ____________________________________

bad_status = [
    "Charged Off",
    "Default",
    "Late (31-120 days)"]

df["target"] = (
    df["loan_status"]
      .isin(bad_status)
      .astype(int))

In [6]:
# Structured Features
# ____________________________________

numerical_features = [

    "loan_amnt",

    "int_rate",

    "annual_inc",

    "dti",

    "revol_util"]

categorical_features = [

    "grade",

    "home_ownership",

    "term"]

text_feature = "title"

In [7]:
# Split data
# ____________________________________

X = df.drop(columns=["target"])

y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(X,

    y,

    test_size=0.20,

    stratify=y,

    random_state=42)

print(X_train.shape)

print(X_test.shape)

(400000, 151)
(100000, 151)


In [8]:
# Text Cleaning for NLP
# ____________________________________

STOP_WORDS = ENGLISH_STOP_WORDS

def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    # Remove numbers
    
    text = re.sub(r"\d+", " ", text)

    # Remove punctuation
    
    text = text.translate(
        str.maketrans(
            "",
            "",
            string.punctuation))

    # Remove extra whitespace
    
    text = re.sub(r"\s+", " ", text)

    # Remove stop words
    
    words = [word
             
             for word in text.split()
             
             if word not in STOP_WORDS]
    
    return " ".join(words)

In [9]:
# create title
# ____________________________________

X_train["clean_title"] = X_train["title"].apply(clean_text)

X_test["clean_title"] = X_test["title"].apply(clean_text)

print("Clean title created.")

Clean title created.


In [10]:
# Text Validation
# ____________________________________

print("TEXT VALIDATION")


print(X_train["clean_title"].isna().sum())

print()

print(X_train["clean_title"].str.len().describe())

TEXT VALIDATION
0

count    400000.000000
mean         17.787923
std           5.064638
min           0.000000
25%          18.000000
50%          18.000000
75%          18.000000
max          30.000000
Name: clean_title, dtype: float64


In [11]:
# Display X_train
# ____________________________________

display(X_train[["title", "clean_title"]].head(10))

,title,clean_title
409428,Credit card refinancing,credit card refinancing
6851,Debt consolidation,debt consolidation
278277,Debt consolidation,debt consolidation
427606,Home improvement,home improvement
149465,Home improvement,home improvement
363478,Debt consolidation,debt consolidation
494216,Debt consolidation,debt consolidation
129594,Debt consolidation,debt consolidation
440797,Car financing,car financing
203761,Debt consolidation,debt consolidation


In [12]:
# Numerical Preprocessing
# _______________________________________________________________

numeric_pipeline = Pipeline([("median_imputer"
                              
                              ,SimpleImputer(strategy="median"))
                             
                             ,("scaler",StandardScaler())])


# Categorical preprocessing
categorical_pipeline = Pipeline([("mode_imputer"
                                  
                                  ,SimpleImputer(strategy="most_frequent"))
                                 
                                 ,("encoder",OneHotEncoder(handle_unknown="ignore"))])

In [13]:
# Combining Both Pipelines
# ____________________________________

structured_preprocessor = ColumnTransformer(

    transformers=[("numeric"
                   ,numeric_pipeline
                   ,numerical_features)
                  
                  ,("categorical"
                    ,categorical_pipeline
                    ,categorical_features)])

In [14]:
# Fitting Preprocessor pipelines only on training data
# _________________________________________________________________

X_train_structured = structured_preprocessor.fit_transform(X_train)

X_test_structured = structured_preprocessor.transform(X_test)

print(X_train_structured.shape)

print(X_test_structured.shape)

(400000, 18)
(100000, 18)


In [15]:
# Text_preprocessing
# ____________________________________

def prepare_text_column(df: pd.DataFrame, text_column: str) -> pd.DataFrame:
    

    df = df.copy()

    df[text_column] = (
        df[text_column]
        .fillna("")
        .astype(str)
        .str.strip())

    return df


def validate_text_column(df: pd.DataFrame, text_column: str) -> None:

    missing = df[text_column].isna().sum()

    if missing > 0:
        raise ValueError(
            f"{text_column} still contains {missing} missing values.")

    print("✓ Text column validation passed.")

#### Saving the pipeline 

In [16]:
# Save preprocessor
# ___________________________________________________

models_dir = Path("models")

models_dir.mkdir(parents=True, exist_ok=True)

# 3. Save your pipeline file inside it
output_path = models_dir / "structured_preprocessor.pkl"

joblib.dump(structured_preprocessor, output_path)

print(f"Structured preprocessing pipeline successfully saved to: {output_path}")

Structured preprocessing pipeline successfully saved to: models/structured_preprocessor.pkl


In [17]:
# Save Processed Feature Matrices
# ____________________________________________________

models_dir = Path("models")

models_dir.mkdir(parents=True, exist_ok=True)

# Save your feature matrices 

joblib.dump(X_train_structured, models_dir / "matrix_A_train.pkl")

joblib.dump(X_test_structured, models_dir / "matrix_A_test.pkl")

print("Matrix A saved successfully.")

Matrix A saved successfully.


In [18]:
# SAVE CLEAN PROCESSED DATASETS
# ____________________________________

processed_dir = Path("data/processed")

processed_dir.mkdir(parents=True, exist_ok=True)


# Save CSV

X_train.to_csv(processed_dir/"X_train.csv",index=False)

X_test.to_csv(processed_dir/"X_test.csv",index=False)

y_train.to_csv(processed_dir/"y_train.csv",index=False)

y_test.to_csv(processed_dir/"y_test.csv",index=False)

# Save Pickle (faster loading)

joblib.dump(X_train,processed_dir/"X_train.pkl")

joblib.dump(X_test,processed_dir/"X_test.pkl")

joblib.dump(y_train,processed_dir/"y_train.pkl")

joblib.dump(y_test,processed_dir/"y_test.pkl")

print("Processed datasets saved successfully.")

Processed datasets saved successfully.


In [19]:
# Text processing report
# __________________________________________________

X_train = prepare_text_column(X_train, "clean_title")

X_test = prepare_text_column(X_test, "clean_title")

validate_text_column(X_train, "clean_title")

validate_text_column(X_test, "clean_title")

✓ Text column validation passed.
✓ Text column validation passed.


In [20]:
# Text Processing report
# ____________________________________________________________

text_report = pd.DataFrame({"Metric":["Training Records",
                                      
                                      "Testing Records",
                                      
                                      "Missing Text (Train)",
                                      
                                      "Missing Text (Test)",
                                      
                                      "Empty Documents (Train)",
                                      
                                      "Empty Documents (Test)",
                                      
                                      "Unique Loan titles",
                                      
                                      "Average Length",
                                      
                                      "Minimum Length",
                                      
                                      "Maximum Length"],
                            
                            "Value":[len(X_train),
                                     
                                     len(X_test),
                                     
                                     X_train["clean_title"].isna().sum(),
                                     
                                     X_test["clean_title"].isna().sum(),
                                     
                                     (X_train["clean_title"].str.strip()=="").sum(),
                                     
                                     (X_test["clean_title"].str.strip()=="").sum(),
                                     
                                     X_train["clean_title"].nunique(),
                                     
                                     round(X_train["clean_title"].str.len().mean(),2),
                                     
                                     X_train["clean_title"].str.len().min(),
                                     
                                     X_train["clean_title"].str.len().max()]})
display(text_report)

# save text report

text_report.to_csv(processed_dir /"text_processing_report.csv",index=False)

,Metric,Value
0,Training Records,400000.00
1,Testing Records,100000.00
2,Missing Text (Train),0.00
3,Missing Text (Test),0.00
4,Empty Documents (Train),21190.00
5,Empty Documents (Test),5234.00
6,Unique Loan titles,24.00
7,Average Length,17.79
8,Minimum Length,0.00
9,Maximum Length,30.00


In [21]:
# Data Transformation 
# ______________________________________________

X_train_A = structured_preprocessor.fit_transform(X_train)

X_test_A = structured_preprocessor.transform(X_test)

In [22]:
print(X_train_A.shape)

(400000, 18)


In [27]:
# Exporting Structured feature names
# ______________________________________________

X_train_A = structured_preprocessor.fit_transform(X_train)

X_test_A = structured_preprocessor.transform(X_test)

structured_feature_names = (structured_preprocessor.get_feature_names_out())

structured_feature_names = pd.DataFrame({

    "Feature_ID": range(len(structured_feature_names)),

    "Feature_Name": structured_feature_names})

# Saved structured feature

structured_feature_names.to_csv(processed_dir/"structured_feature_names.csv",index=False)